# 01 · Ingestión y transformación del dataset (EA1)

Flujo: **Hugging Face → `HFDatasetLoader` → DataFrame → transformaciones → `DatasetSummarizer` → `reports/data-summary.txt`**.

Dataset: [`Nicolybgs/healthcare_data`](https://huggingface.co/datasets/Nicolybgs/healthcare_data). Ejecuta las celdas en orden.

## 1. Preparar el entorno e importar las clases

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))  # permite `import src...` sin rutas absolutas

import pandas as pd
pd.set_option("display.max_columns", 30)
from src import config
from src.data.huggingface_loader import HFDatasetLoader
from src.data.dataset_io import make_loader, save_csv_capped, build_metadata
from src.data import preprocessing as prep
from src.utils.reporter import DatasetSummarizer, save_json

/opt/hostedtoolcache/Python/3.12.15/x64/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Verificar configuración y splits reales, y cargar el dataset

`available_splits()` consulta Hugging Face, así que no se asume nada. Se descarga el dataset **completo** (sin muestreo): la limpieza, el resumen y los modelos trabajan con todas las filas. Solo los CSV que se guardan en el repositorio se limitan de tamaño (ver sección 4).

In [2]:
loader = make_loader()   # HFDatasetLoader(dataset_id, config, split) sin max_rows
print("Configuraciones y splits disponibles:", loader.available_splits())
raw = loader.load()
loader.info()

Configuraciones y splits disponibles: {'default': ['train']}


{'dataset_id': 'Nicolybgs/healthcare_data',
 'config': 'default',
 'split': 'train',
 'filas_originales_en_hf': 500000,
 'filas_cargadas': 500000,
 'columnas': ['Available Extra Rooms in Hospital',
  'Department',
  'Ward_Facility_Code',
  'doctor_name',
  'staff_available',
  'patientid',
  'Age',
  'gender',
  'Type of Admission',
  'Severity of Illness',
  'health_conditions',
  'Visitors with Patient',
  'Insurance',
  'Admission_Deposit',
  'Stay (in days)'],
 'muestreo': 'dataset completo'}

## 3. Estructura del DataFrame original: tamaño, tipos y muestra

In [3]:
print(f"Tamaño: {raw.shape[0]} filas x {raw.shape[1]} columnas")
display(raw.dtypes.rename("tipo").to_frame())
display(raw.isna().sum().rename("nulos").to_frame().query("nulos > 0"))
raw.head()

Tamaño: 500000 filas x 15 columnas


,tipo
Available Extra Rooms in Hospital,int64
Department,object
Ward_Facility_Code,object
doctor_name,object
staff_available,int64
patientid,int64
Age,object
gender,object
Type of Admission,object
Severity of Illness,object


,nulos
health_conditions,151888


,Available Extra Rooms in Hospital,Department,Ward_Facility_Code,doctor_name,staff_available,patientid,Age,gender,Type of Admission,Severity of Illness,health_conditions,Visitors with Patient,Insurance,Admission_Deposit,Stay (in days)
0,4,gynecology,D,Dr Sophia,0,33070,41-50,Female,Trauma,Extreme,Diabetes,4,Yes,2966.408696,8
1,4,gynecology,B,Dr Sophia,2,34808,31-40,Female,Trauma,Minor,Heart disease,2,No,3554.835677,9
2,2,gynecology,B,Dr Sophia,8,44577,21-30,Female,Trauma,Extreme,Diabetes,2,Yes,5624.733654,7
3,4,gynecology,D,Dr Olivia,7,3695,31-40,Female,Urgent,Moderate,None,4,No,4814.149231,8
4,2,anesthesia,E,Dr Mark,10,108956,71-80,Male,Trauma,Moderate,Diabetes,2,No,5169.269637,34


## 4. Guardar los datos sin transformar (`data/raw`)
El DataFrame en memoria conserva todas las filas. Al guardar el CSV se aplica el límite `CSV_MAX_ROWS` de `src/config.py` (muestra aleatoria reproducible, `seed=42`) para no exceder el peso permitido por GitHub.

In [4]:
saved_raw = save_csv_capped(raw, config.RAW_CSV)
saved_raw

{'archivo': 'data/raw/healthcare_raw.csv',
 'filas_guardadas': 100000,
 'filas_totales': 500000,
 'tamano_mb': 8.95}

## 5. Transformaciones

Cada transformación devuelve un DataFrame nuevo y se muestra su efecto. El orden coincide con `preprocess()`.

### 5.1 Tratar nulos y duplicados de forma explícita
* `health_conditions` tiene nulos que en este dataset significan **"sin condición preexistente"**; se rellenan con la categoría `"None"` en lugar de borrar ~30 % de las filas.
* Se eliminan filas duplicadas **exactas** (antes de quitar `patientid`, para no tratar como repetidos a pacientes distintos con características iguales).

In [5]:
df = prep.strip_text_columns(raw)   # limpieza menor: espacios sobrantes
dup_antes = int(df.duplicated().sum())
df1 = prep.drop_duplicates(df)
nulos_antes = int(df1["health_conditions"].isna().sum())
df1 = prep.fill_missing_conditions(df1)
print(f"Duplicados eliminados: {dup_antes} | filas: {len(df)} -> {len(df1)}")
print(f"Nulos en health_conditions: {nulos_antes} -> {int(df1['health_conditions'].isna().sum())}")
df1["health_conditions"].value_counts().to_frame()

Duplicados eliminados: 0 | filas: 500000 -> 500000
Nulos en health_conditions: 151888 -> 0


,count
health_conditions,
None,151888
Other,94411
High Blood Pressure,79402
Diabetes,73644
Asthama,65514
Heart disease,35141


### 5.2 Eliminar `patientid`
Es un identificador único: no tiene valor predictivo y podría inducir al modelo a memorizar pacientes. Se elimina de la versión analítica.

In [6]:
df2 = prep.drop_identifier(df1)
print("patientid" in df2.columns, "|", df1.shape, "->", df2.shape)

False | (500000, 15) -> (500000, 14)


### 5.3 Convertir `Age` de rangos de texto a número
`Age` viene como rangos (`"41-50"`). Los modelos lineales necesitan un valor numérico: se usa el **punto medio** del rango (`45.5`) en la nueva columna `age_midpoint`, que reemplaza a `Age`.

In [7]:
df3 = prep.encode_age_midpoint(df2)
display(df2["Age"].value_counts().sort_index().to_frame())
df3[["age_midpoint"]].drop_duplicates().sort_values("age_midpoint").T

,count
Age,
0-10,3368
11-20,46536
21-30,159793
31-40,133373
41-50,80406
51-60,21718
61-70,26556
71-80,18703
81-90,8181


,534,11,2,1,0,18,35,4,212,426
age_midpoint,5.0,15.5,25.5,35.5,45.5,55.5,65.5,75.5,85.5,95.5


### 5.4 Normalizar nombres de columnas a `snake_case`
Nombres como `Stay (in days)` o `Available Extra Rooms in Hospital` son incómodos de usar en código. Se convierten a minúsculas con guiones bajos (`stay_in_days`).

In [8]:
processed = prep.normalize_column_names(df3)
pd.DataFrame({"antes": list(df3.columns), "después": list(processed.columns)})

,antes,después
0,Available Extra Rooms in Hospital,available_extra_rooms_in_hospital
1,Department,department
2,Ward_Facility_Code,ward_facility_code
3,doctor_name,doctor_name
4,staff_available,staff_available
5,gender,gender
6,Type of Admission,type_of_admission
7,Severity of Illness,severity_of_illness
8,health_conditions,health_conditions
9,Visitors with Patient,visitors_with_patient


### 5.5 Comprobación y guardado limitado (`data/processed`)
La función `preprocess()` aplica todos los pasos en el mismo orden; se verifica que da el mismo resultado.

In [9]:
assert processed.equals(prep.preprocess(raw)), "preprocess() debe coincidir con los pasos manuales"
saved_processed = save_csv_capped(processed, config.PROCESSED_CSV)  # mismo límite de filas
print(f"Filas en memoria: {len(processed)} | filas guardadas en CSV: {saved_processed['filas_guardadas']} ({saved_processed['tamano_mb']} MB)")

Filas en memoria: 500000 | filas guardadas en CSV: 100000 (8.35 MB)


## 6. Resumen del DataFrame **transformado** con `DatasetSummarizer`

In [10]:
ruta = DatasetSummarizer().save(processed, config.DATA_SUMMARY_TXT)
metadata = build_metadata(loader.info(), raw, processed, [saved_raw, saved_processed])
save_json(metadata, config.INGESTION_METADATA_JSON)
print("Resumen generado en:", ruta.relative_to(ROOT))
print(ruta.read_text(encoding="utf-8"))

Resumen generado en: reports/data-summary.txt
RESUMEN DEL DATAFRAME
Filas: 500000
Columnas: 14
Filas duplicadas: 0

Nombres de columnas:
available_extra_rooms_in_hospital, department, ward_facility_code, doctor_name, staff_available, gender, type_of_admission, severity_of_illness, health_conditions, visitors_with_patient, insurance, admission_deposit, stay_in_days, age_midpoint

Tipos de datos:
- available_extra_rooms_in_hospital: int64
- department: object
- ward_facility_code: object
- doctor_name: object
- staff_available: int64
- gender: object
- type_of_admission: object
- severity_of_illness: object
- health_conditions: object
- visitors_with_patient: int64
- insurance: object
- admission_deposit: float64
- stay_in_days: int64
- age_midpoint: float64

Valores nulos por columna:
- available_extra_rooms_in_hospital: 0
- department: 0
- ward_facility_code: 0
- doctor_name: 0
- staff_available: 0
- gender: 0
- type_of_admission: 0
- severity_of_illness: 0
- health_conditions: 0
- vis

## 7. Conclusión
Se aplicaron cinco transformaciones (eliminar identificador, `Age` → numérica, `snake_case`, relleno explícito de nulos y eliminación de duplicados). El `.txt` describe el DataFrame **transformado completo**; los CSV `raw` y `processed` en `data/` son una muestra reproducible limitada por peso.